### Import pysammos

In [1]:
import os
os.environ["NUMBA_NUM_THREADS"] = "8"
print(f">>> Numba is using {os.environ['NUMBA_NUM_THREADS']} cores")

>>> Numba is using 8 cores


In [2]:
import numpy as np

In [3]:
from pysammos.utils.config_loader import load_config
from pysammos.coarse_graining import CoarseGraining

Hello from pysammos
Loading macroscopic_fields package...
Loading data_read package...
Loading particle_phase package...
Loading spatial_weights package...
Loading neighbour_search package...
Loading grid_generation package...
Loading data_handle package...
Loading data_write package...


### Run Coarse - Graining Workflow

Initialise the Coarse Graining class

In [4]:
# Load the configuration from the ini file
cfg = load_config("config.ini")  
print("-------------------- config.ini file read --------------------")
# Initialize the CoarseGraining class with the loaded configuration
CG = CoarseGraining(
    particle_path=cfg["particles_path"],
    contacts_path=cfg["contacts_path"],
    output_path=cfg["output_path"],
    start_timestep=cfg["t0"],
    end_timestep=cfg["tf"],
    dt_time_step=cfg["dt"],
    DEM_keymap=cfg["key_mapping"],
    grid_info=cfg["grid_info"],
    weight_function=cfg["smoothing_function"],
    fields_to_export=cfg["fields_to_export"],
    ignore_phases=cfg["partialignore"],
    h5_output=cfg["h5_output"],
    vkthdf_output=cfg["vkthdf_output"],
                    ) 
print("  ") ; print("-------------------- CoarseGraining class initialised --------------------")

-------------------- config.ini file read --------------------
Output path already exists: ./PysammosCG/
  
-------------------- CoarseGraining class initialised --------------------


Option (A) Run all at once:

In [ ]:
#CG.run()

Option (B) Run step by step:

In [5]:
# 1. Load the size-relevant particle data for the first time step
Bounds_t0, Diameter_t0, Density_t0, Mass_t0, GlobalID_t0 = CG.data_sampling()

XML-based PolyData detected.


In [6]:
# 2. Calculate the particle size range
CG.get_particle_size_statistics(Diameter_t0, Mass_t0)
print(">> Particle size statistics: ") 
print("       d43: ", CG.d43)
print("       dmax: ", CG.dmax)
print("       d50: ", CG.d50)
print("       d32: ", CG.d32)
print("       drms: ", CG.drms)

>> Particle size statistics: 
       d43:  0.01651367411185649
       dmax:  0.019993
       d50:  0.016306963
       d32:  0.016109877148148044
       drms:  0.015422490500786803


In [7]:
# 3. Get the phases
CG.get_particle_phases(Diameter_t0, Density_t0, GlobalID_t0, 8)
print(">> Phases: ")
print("       Diameters: ", CG.phases[:,0])
print("       Densities: ", CG.phases[:,1])

Ignoring phases. Using mean density and d50 as phase.
>> Phases: 
       Diameters:  [0.01630696]
       Densities:  [2700.]


In [8]:
# 4. Calculate the CG grid spacing
CG.set_resolution(CG.d43) # here you can input different number, to make w and c bigger or smaller 
print(">> Coarse Graining resolution: ")
print("       c:", CG.c)
print("       w:", CG.w)

>> Coarse Graining resolution: 
       c: 0.024770511167784733
       w: 0.012385255583892366


In [9]:
# 5. Generate the CG grid
CG.generate_grid()
print(">> Grid: ")
print("       Grid Points: ", CG.GridPoints.shape, "First Point: ", CG.GridPoints[0])
print("       Nodes: ", CG.Nodes)
print("       Spacing: ", CG.Spacing)

Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [0.00105, 0.5], y = [0.001, 0.24], z = [None, None], x_transect = None, y_transect = None, z_transect = 0.0
>> Grid: 
       Grid Points:  (882, 3) First Point:  [0.00105 0.001   0.     ]
       Nodes:  [42 21  1]
       Spacing:  [0.01216951 0.01195   ]


In [10]:
# 6. Calculate the CG fields
CG.fields_in_time()

 
-------------------- Calculating Coarse Grained Fields --------------------
 
---> Time step 0: time 0150 ................................................
Loading data ... 
  Particle data loaded
type of output (e.g., inter, float etc, 64, 32):  float32 float32 float32 float32
  Repeated pairs in contact data:  1142
  Contact data loaded and mapped
  Coordination number not required. Skipping its calculation.
... data loaded
Matching particles to grid points ...
... particles assigned to grid nodes
Computing weights ...
  Using Lucy kernel with cutoff 0.024770511167784733 and search sampling factor 1000
... weights computed
Computing Coarse Graining fields...
  volume fraction done
  mixture density done
  momentum density done
  contact tensor done


TBB Warning: The number of workers is currently limited to 1. The request for 7 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



  total stress done
... fields computed
Writing results for timestep 150...
    New node dimensions: (np.int64(42), np.int64(21), 2)
    New node spacing: (np.float64(0.012169512195121952), np.float64(0.011949999999999999), 1e-08)
ox, oy, oz = 42, 21, 1
value shape before promotion: (882,)
arr shape after reshape to 2D: (42, 21, 1)
arr shape after promotion to thin 3D: (42, 21, 2)
returning scalar promoted data with shape: (1764,)
ox, oy, oz = 42, 21, 1
value shape before promotion: (882, 3)
arr shape after reshape to 2D: (42, 21, 1, 3)
arr shape after promotion to thin 3D: (42, 21, 2, 3)
returning vector/tensor promoted data with shape: (1764, 3)
ox, oy, oz = 42, 21, 1
value shape before promotion: (882, 2, 2)
arr shape after reshape to 2D: (42, 21, 1, 4)
arr shape after promotion to thin 3D: (42, 21, 2, 4)
returning tensor promoted data with shape: (1764, 1, 4)
ox, oy, oz = 42, 21, 1
value shape before promotion: (882,)
arr shape after reshape to 2D: (42, 21, 1)
arr shape after promo

In [ ]:
a = 0.3
list = [2,3]
list = [a] + list
print(list)

In [ ]:
value = np.array([1, 2, 3, 4])

arr = np.concatenate([value, value], axis=0)

print(arr)

In [ ]:
array = np.array([[1, 2, 3], [1, 2, 3], [1, 2, 3], [1, 2, 3]])

array_reshaped_F = array.reshape((3,4), order='F')
array_reshaped_C = array.reshape((3,4), order='C')

print("Original array:")
print(array)
print("Reshaped array F:")
print(array_reshaped_F)
print("Reshaped array C:")
print(array_reshaped_C)

### Example of sweep of w/d values

In [ ]:
# Load the configuration from the ini file
cfg = load_config("config.ini")  
print("-------------------- config.ini file read --------------------")
# Initialize the CoarseGraining class with the loaded configuration
CG = CoarseGraining(
    particle_path=cfg["particles_path"],
    contacts_path=cfg["contacts_path"],
    output_path=cfg["output_path"],
    start_timestep=cfg["t0"],
    end_timestep=cfg["tf"],
    dt_time_step=1,
    DEM_keymap=cfg["key_mapping"],
    grid_info=cfg["grid_info"],
    weight_function='Lucy',
    fields_to_export=cfg["fields_to_export"],
    ignore_phases=cfg["partialignore"],
    h5_output=cfg["h5_output"],
    vkthdf_output=cfg["vkthdf_output"],
                    ) 
print("  ") ; print("-------------------- CoarseGraining class initialised --------------------")

In [ ]:
CG.sweep_CG_widths(w_d=np.array([0.5, 0.8, 1.1]))
print(">> Sweep of CG widths completed.")
print(CG.GridPoints)